In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
            
db_candidates = [
    Path.cwd() / "data_handling" / "sepsis.db",
    Path.cwd().parent / "sepsis.db",
    Path.cwd().parent.parent / "data_handling" / "sepsis.db",
]

DB_PATH = next((path for path in db_candidates if path.exists() and path.stat().st_size > 0), None)
if DB_PATH is None:
    raise FileNotFoundError("Could not find the main sepsis.db database.")

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")


Connected to: c:\Users\KRISH JHA\OneDrive\Desktop\Sepsis\model\data_handling\sepsis.db


In [55]:
# Start from the raw table each time so the cleaning pipeline is reproducible.
df = pd.read_sql_query("SELECT * FROM patient_data;", conn)

print("Raw shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Total missing values:", int(df.isna().sum().sum()))


Raw shape: (1552318, 42)
Duplicate rows: 108
Total missing values: 43515455


In [59]:
# Remove exact duplicate rows.
cursor=conn.cursor()
cursor.execute("""
    CREATE TABLE clean_patient_data AS
    SELECT DISTINCT *
    FROM patient_data;
    """)

df = pd.read_sql_query("SELECT * FROM clean_patient_data;", conn)
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after duplicate removal:", df.shape)


Shape after duplicate removal: (1552210, 42)


In [60]:
# Convert feature columns to numeric. Patient_id stays as text.
for col in df.columns:
    if col != "Patient_id":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Sort each patient's ICU time series before forward/backward filling.
df = df.sort_values(["Patient_id", "ICULOS"]).reset_index(drop=True)


In [61]:
missing_percentage = (df.isna().mean() * 100).sort_values(ascending=False)

low_missing = missing_percentage[(missing_percentage > 0) & (missing_percentage < 20)].index.tolist()
medium_missing = missing_percentage[(missing_percentage >= 20) & (missing_percentage < 95)].index.tolist()
high_missing = missing_percentage[missing_percentage >= 95].index.tolist()

print("Low missing (<20%):")
print(low_missing)

print()
print("Medium missing (20-95%):")
print(medium_missing)

print()
print("High missing (>=95%):")
print(high_missing)


Low missing (<20%):
['Resp', 'SBP', 'O2Sat', 'MAP', 'HR', 'HospAdmTime']

Medium missing (20-95%):
['BaseExcess', 'PaCO2', 'Calcium', 'Platelets', 'Creatinine', 'Magnesium', 'WBC', 'BUN', 'pH', 'Hgb', 'FiO2', 'Hct', 'Potassium', 'Glucose', 'Temp', 'Unit2', 'Unit1', 'DBP']

High missing (>=95%):
['Bilirubin_direct', 'Fibrinogen', 'TroponinI', 'Bilirubin_total', 'Alkalinephos', 'AST', 'Lactate', 'PTT', 'SaO2', 'EtCO2', 'Phosphate', 'HCO3', 'Chloride']


In [62]:
valid_ranges = {
    # Vital Signs
    "HR": (20, 220),
    "O2Sat": (50, 100),
    "Temp": (25.0, 43.5),
    "SBP": (40, 300),
    "MAP": (20, 200),
    "DBP": (20, 180),
    "Resp": (4, 80),
    "EtCO2": (10, 100),

    # Blood Gas
    "BaseExcess": (-30, 30),
    "HCO3": (5, 45),
    "FiO2": (0.21, 1.0),
    "pH": (6.8, 7.8),
    "PaCO2": (10, 120),
    "SaO2": (50, 100),

    # Chemistry
    "AST": (0, 5000),
    "BUN": (1, 200),
    "Alkalinephos": (10, 2000),
    "Calcium": (4, 20),
    "Chloride": (60, 140),
    "Creatinine": (0.1, 20),
    "Bilirubin_direct": (0, 20),
    "Glucose": (20, 1000),
    "Lactate": (0.2, 30),
    "Magnesium": (0.5, 8),
    "Phosphate": (0.5, 15),
    "Potassium": (1.5, 9),
    "Bilirubin_total": (0, 40),
    "TroponinI": (0, 100),

    # Hematology
    "Hct": (10, 70),
    "Hgb": (3, 25),
    "PTT": (15, 200),
    "WBC": (0.1, 200),
    "Fibrinogen": (50, 1000),
    "Platelets": (1, 1500),

    # Demographics and labels
    "Age": (0, 100),
    "Gender": (0, 1),
    "Unit1": (0, 1),
    "Unit2": (0, 1),
    "HospAdmTime": (-5000, 0),
    "ICULOS": (1, 1000),
    "SepsisLabel": (0, 1),
}

invalid_counts = {}
for col, (min_value, max_value) in valid_ranges.items():
    if col in df.columns:
        invalid = df[col].notna() & ((df[col] < min_value) | (df[col] > max_value))
        invalid_counts[col] = int(invalid.sum())
        df.loc[invalid, col] = np.nan

pd.Series(invalid_counts, name="invalid_values").sort_values(ascending=False)


Calcium             9997
HospAdmTime         8773
Resp                1837
MAP                  444
FiO2                 395
O2Sat                272
SaO2                 271
AST                  253
TroponinI            190
DBP                  151
PTT                  141
SBP                  114
Potassium             74
Fibrinogen            39
Creatinine            38
HCO3                  38
Bilirubin_total       32
Bilirubin_direct      28
WBC                   18
Phosphate             18
BUN                   16
Magnesium             13
Alkalinephos          13
pH                    12
Glucose               10
Hct                   10
Hgb                    8
Temp                   6
HR                     6
BaseExcess             5
Chloride               4
Platelets              4
Lactate                1
EtCO2                  0
PaCO2                  0
Gender                 0
Age                    0
Unit1                  0
Unit2                  0
ICULOS                 0


In [63]:
# Remove columns that are almost entirely missing. These add noise and are hard
# to impute reliably for a first sepsis prediction model.
protected_columns = {"Patient_id", "ICULOS", "SepsisLabel"}
columns_to_drop = [col for col in high_missing if col not in protected_columns]

df = df.drop(columns=columns_to_drop)
print("Dropped high-missing columns:")
print(columns_to_drop)
print("Shape after dropping high-missing columns:", df.shape)


Dropped high-missing columns:
['Bilirubin_direct', 'Fibrinogen', 'TroponinI', 'Bilirubin_total', 'Alkalinephos', 'AST', 'Lactate', 'PTT', 'SaO2', 'EtCO2', 'Phosphate', 'HCO3', 'Chloride']
Shape after dropping high-missing columns: (1552210, 29)


In [64]:
# Add missingness indicators before imputation. This lets the model learn that
# "not measured" can itself be clinically meaningful.
feature_columns = [col for col in df.columns if col not in ["Patient_id", "SepsisLabel"]]
columns_with_missing = [col for col in feature_columns if df[col].isna().any()]

for col in columns_with_missing:
    df[f"{col}_missing"] = df[col].isna().astype("int8")

print(f"Added {len(columns_with_missing)} missing-indicator columns.")


Added 24 missing-indicator columns.


In [65]:
# Fill time-varying measurements within each patient.
# Forward fill uses earlier measurements; backward fill handles the first rows
# before a measurement appears for that patient.
impute_columns = [col for col in feature_columns if col in df.columns]

df[impute_columns] = df.groupby("Patient_id", sort=False)[impute_columns].ffill()
df[impute_columns] = df.groupby("Patient_id", sort=False)[impute_columns].bfill()


In [66]:
# Fill any values still missing after patient-level fills.
# These are usually patients who never had that lab/vital recorded.
remaining_missing_before = int(df.isna().sum().sum())

numeric_columns = [col for col in df.select_dtypes(include=[np.number]).columns if col != "SepsisLabel"]
for col in numeric_columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

if df["Patient_id"].isna().any():
    df["Patient_id"] = df["Patient_id"].fillna("unknown_patient")

remaining_missing_after = int(df.isna().sum().sum())
print("Missing values before final median fill:", remaining_missing_before)
print("Missing values after final fill:", remaining_missing_after)


Missing values before final median fill: 5512222
Missing values after final fill: 0


In [67]:
print("Final shape:", df.shape)
print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Total missing values after cleaning:", int(df.isna().sum().sum()))

df.head()


Final shape: (1552210, 53)
Duplicate rows after cleaning: 0
Total missing values after cleaning: 0


,Patient_id,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,BaseExcess,FiO2,...,Glucose_missing,Magnesium_missing,Potassium_missing,Hct_missing,Hgb_missing,WBC_missing,Platelets_missing,Unit1_missing,Unit2_missing,HospAdmTime_missing
0,p000001,97.0,95.0,36.11,98.0,75.33,62.0,19.0,24.0,0.28,...,1,1,1,1,1,1,1,1,1,0
1,p000001,97.0,95.0,36.11,98.0,75.33,62.0,19.0,24.0,0.28,...,1,1,1,1,1,1,1,1,1,0
2,p000001,89.0,99.0,36.11,122.0,86.00,62.0,22.0,24.0,0.28,...,1,1,1,1,1,1,1,1,1,0
3,p000001,90.0,95.0,36.11,122.0,86.00,62.0,30.0,24.0,0.28,...,1,1,1,1,1,1,1,1,1,0
4,p000001,103.0,88.5,36.11,122.0,91.33,62.0,24.5,24.0,0.28,...,1,1,1,1,1,1,1,1,1,0


In [68]:
# Save the cleaned data using one correctly spelled table name.
df.to_sql("clean_patient_data", con=conn, if_exists="replace", index=False)
conn.commit()

print("Saved cleaned data to table: clean_patient_data")


Saved cleaned data to table: clean_patient_data


In [69]:
# Quick database check.
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn,
)

print(tables)
print(pd.read_sql_query("SELECT COUNT(*) AS rows FROM clean_patient_data;", conn))


                 name
0  clean_patient_data
1        patient_data
      rows
0  1552210


In [70]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1552210 entries, 0 to 1552209
Data columns (total 53 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   Patient_id           1552210 non-null  str    
 1   HR                   1552210 non-null  float64
 2   O2Sat                1552210 non-null  float64
 3   Temp                 1552210 non-null  float64
 4   SBP                  1552210 non-null  float64
 5   MAP                  1552210 non-null  float64
 6   DBP                  1552210 non-null  float64
 7   Resp                 1552210 non-null  float64
 8   BaseExcess           1552210 non-null  float64
 9   FiO2                 1552210 non-null  float64
 10  pH                   1552210 non-null  float64
 11  PaCO2                1552210 non-null  float64
 12  BUN                  1552210 non-null  float64
 13  Calcium              1552210 non-null  float64
 14  Creatinine           1552210 non-null  float64
 15  Glucose  

In [2]:
df=pd.read_sql_query("""
    Select * from clean_patient_data
""",conn)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1552210 entries, 0 to 1552209
Data columns (total 53 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   Patient_id           1552210 non-null  str    
 1   HR                   1552210 non-null  float64
 2   O2Sat                1552210 non-null  float64
 3   Temp                 1552210 non-null  float64
 4   SBP                  1552210 non-null  float64
 5   MAP                  1552210 non-null  float64
 6   DBP                  1552210 non-null  float64
 7   Resp                 1552210 non-null  float64
 8   BaseExcess           1552210 non-null  float64
 9   FiO2                 1552210 non-null  float64
 10  pH                   1552210 non-null  float64
 11  PaCO2                1552210 non-null  float64
 12  BUN                  1552210 non-null  float64
 13  Calcium              1552210 non-null  float64
 14  Creatinine           1552210 non-null  float64
 15  Glucose  